# Build temporal modeling dataset

Create the financial indicators and the exact one-year-ahead target used in the modeling experiment.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv('../data/processed/financial_sporting_data.csv')
modeling = df.copy()

## Check club-year key

In [2]:
duplicates = modeling[modeling.duplicated(subset=['club', 'fiscal_year'], keep=False)]

assert duplicates.empty
duplicates

,club,fiscal_year,cash,current_assets,total_assets,current_liabilities,total_liabilities,equity,net_result,division,...,league_size,promoted,relegated,table_source,table_source_url,outcome_source,notes,points_per_game,goal_difference_per_game,relative_league_position


## Create financial features at year t

In [3]:
modeling['current_liquidity'] = (modeling['current_assets']
    / modeling['current_liabilities'].replace(0, np.nan)
)

modeling['liability_ratio'] = (
    modeling['total_liabilities']
    / modeling['total_assets'].replace(0, np.nan)
)

modeling['return_on_assets'] = (
    modeling['net_result']
    / modeling['total_assets'].replace(0, np.nan)
)

modeling['loss_current_year'] = (
    modeling['net_result'] < 0
).astype(int)

In [4]:
financial_features = ['current_liquidity', 'liability_ratio', 'return_on_assets',]

modeling[financial_features].isna().sum()

current_liquidity    0
liability_ratio      0
return_on_assets     0
dtype: int64

## Create exact t → t+1 target

The target must come from the immediately following fiscal year. A later available observation cannot substitute for a missing year.

In [5]:
next_year = df[['club', 'fiscal_year', 'net_result']].copy()

next_year = next_year.rename(columns={
    'fiscal_year': 'target_year',
    'net_result': 'net_result_next_year',
})

next_year['fiscal_year'] = next_year['target_year'] - 1

modeling = modeling.merge(
    next_year,
    how='left',
    on=['club', 'fiscal_year'],
    validate='one_to_one',
)

In [6]:
available_target = modeling['net_result_next_year'].notna()

modeling['loss_next_year'] = pd.NA
modeling.loc[available_target, 'loss_next_year'] = (
    modeling.loc[available_target, 'net_result_next_year'] < 0
).astype(int)

modeling['loss_next_year'] = modeling['loss_next_year'].astype('Int64')

## Audit unavailable targets

In [7]:
without_target = (
    modeling.loc[
        modeling['net_result_next_year'].isna(),
        ['club', 'fiscal_year'],
    ]
    .sort_values(['fiscal_year', 'club'])
    .reset_index(drop=True)
)

without_target.groupby('fiscal_year').size()

fiscal_year
2016     1
2018     1
2023     1
2024     1
2025    33
dtype: int64

In [8]:
without_target[without_target['fiscal_year'] < modeling['fiscal_year'].max()]

,club,fiscal_year
0,Vitória,2016
1,Mirassol,2018
2,Figueirense,2023
3,CSA,2024


## Keep labeled one-year transitions

In [9]:
modeling = modeling[modeling['net_result_next_year'].notna()].copy()

modeling['loss_next_year'] = modeling['loss_next_year'].astype(int)

assert (modeling['target_year'] == modeling['fiscal_year'] + 1).all()
assert modeling['loss_next_year'].isin([0, 1]).all()
assert not modeling.duplicated(subset=['club', 'fiscal_year']).any()

modeling.shape

(302, 34)

## Target distribution

In [10]:
modeling['loss_next_year'].value_counts().sort_index()

loss_next_year
0    134
1    168
Name: count, dtype: int64

In [11]:
modeling.groupby('fiscal_year')['loss_next_year'].agg(
    observations='count',
    losses='sum',
    loss_rate='mean',
)

,observations,losses,loss_rate
fiscal_year,,,
2016,32,18,0.562500
2017,33,16,0.484848
2018,32,17,0.531250
2019,33,24,0.727273
2020,35,17,0.485714
2021,35,15,0.428571
2022,35,14,0.400000
2023,34,25,0.735294
2024,33,22,0.666667


## Sporting availability in the labeled sample

Rows without a national division remain in the general modeling dataset. The main financial vs. financial + sporting comparison will later use the common sample.

In [12]:
sporting_features = [
    'division',
    'points_per_game',
    'goal_difference_per_game',
    'relative_league_position',
    'promoted',
    'relegated',
]

modeling.loc[
    modeling[sporting_features].isna().any(axis=1),
    ['club', 'fiscal_year'] + sporting_features,
]

,club,fiscal_year,division,points_per_game,goal_difference_per_game,relative_league_position,promoted,relegated
226,Mirassol,2016,NaN,NaN,NaN,NaN,NaN,NaN
227,Mirassol,2017,NaN,NaN,NaN,NaN,NaN,NaN
252,Paraná Clube,2023,NaN,NaN,NaN,NaN,NaN,NaN
253,Paraná Clube,2024,NaN,NaN,NaN,NaN,NaN,NaN
290,Santa Cruz,2024,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
common_sample = modeling.dropna(subset=sporting_features)

pd.Series({
    'all_labeled_observations': len(modeling),
    'common_financial_sporting_sample': len(common_sample),
})

all_labeled_observations            302
common_financial_sporting_sample    297
dtype: int64

## 2020 sensitivity checkpoint

The main dataset retains 2020. The modeling experiment will later be repeated without observations whose feature year is 2020.

In [14]:
pd.Series({
    'all_labeled_observations': len(modeling),
    'without_feature_year_2020': len(modeling[modeling['fiscal_year'] != 2020]),
})

all_labeled_observations     302
without_feature_year_2020    267
dtype: int64

## Export dataset

In [15]:
output_path = Path('../data/processed/modeling_data_v1.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

modeling.to_csv(output_path, index=False)

modeling.shape

(302, 34)